In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from umap import UMAP

In [ ]:
CHECKPOINT = "./outputs/1b_layer16_k512_ef16_auxk512_ep40_best/checkpoints/checkpoint_final.pt"

ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state):
    state = {k.removeprefix("module."): v for k, v in state.items()}

W = state["decoder.weight"].float().numpy()  # (input_dim, n_features)
W = (W / np.linalg.norm(W, axis=0, keepdims=True)).T  # (n_features, input_dim), unit norm
print(f"Feature vectors: {W.shape}")

## UMAP Hyperparameter Sweep

Explore different UMAP parameters to find the best layout for your features.

In [ ]:
n_neighbors_vals = [10, 15, 30, 50]
min_dist_vals = [0.05, 0.1, 0.25, 0.5]

results = {}
for nn in n_neighbors_vals:
    for md in min_dist_vals:
        print(f"n_neighbors={nn}, min_dist={md}")
        coords = UMAP(
            n_neighbors=nn, min_dist=md,
            metric="cosine", random_state=42,
        ).fit_transform(W)
        results[(nn, md)] = coords

In [ ]:
fig, axes = plt.subplots(
    len(n_neighbors_vals), len(min_dist_vals),
    figsize=(4 * len(min_dist_vals), 4 * len(n_neighbors_vals)),
)

for i, nn in enumerate(n_neighbors_vals):
    for j, md in enumerate(min_dist_vals):
        ax = axes[i, j]
        coords = results[(nn, md)]
        ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.3)
        ax.set_title(f"nn={nn}, md={md}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

## Save chosen UMAP params

Once you've picked the best `(n_neighbors, min_dist)`, set them below and run.

In [ ]:
# Set your chosen params here
BEST_NN = 15
BEST_MD = 0.1

best_coords_2d = results[(BEST_NN, BEST_MD)]
print(f"Chosen UMAP params: n_neighbors={BEST_NN}, min_dist={BEST_MD}")
print(f"2D coordinates shape: {best_coords_2d.shape}")

## HDBSCAN Clustering Sweep

Now explore HDBSCAN parameters on the chosen UMAP embedding. We compute a higher-dimensional UMAP first (as in the original code) to preserve more structure for clustering.

In [ ]:
# Compute higher-dimensional UMAP for clustering (preserves more structure)
print(f"Computing 10D UMAP for clustering...")
umap_cluster = UMAP(
    n_components=10,
    n_neighbors=BEST_NN,
    min_dist=BEST_MD,
    metric="cosine",
    random_state=42,
)
coords_high = umap_cluster.fit_transform(W)
print(f"High-dim coordinates shape: {coords_high.shape}")

In [ ]:
import hdbscan

min_cluster_sizes = [10, 20, 50, 100, 200]
cluster_results = {}

for mcs in min_cluster_sizes:
    print(f"HDBSCAN min_cluster_size={mcs}")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        metric="euclidean",
    )
    cluster_ids = clusterer.fit_predict(coords_high)
    n_clusters = len(set(cluster_ids)) - (1 if -1 in cluster_ids else 0)
    n_noise = np.sum(cluster_ids == -1)
    print(f"  → {n_clusters} clusters, {n_noise} noise points")
    cluster_results[mcs] = cluster_ids

In [ ]:
# Visualize clustering results
fig, axes = plt.subplots(1, len(min_cluster_sizes), figsize=(5 * len(min_cluster_sizes), 5))

for i, mcs in enumerate(min_cluster_sizes):
    ax = axes[i]
    cluster_ids = cluster_results[mcs]
    
    # Color by cluster (noise points in gray)
    colors = cluster_ids.copy().astype(float)
    colors[cluster_ids == -1] = np.nan  # noise points
    
    scatter = ax.scatter(
        best_coords_2d[:, 0], best_coords_2d[:, 1],
        c=cluster_ids, s=1, alpha=0.6, cmap="tab20"
    )
    
    # Mark noise points
    noise_mask = cluster_ids == -1
    if np.any(noise_mask):
        ax.scatter(
            best_coords_2d[noise_mask, 0], best_coords_2d[noise_mask, 1],
            c="gray", s=0.5, alpha=0.3, label="Noise"
        )
    
    ax.set_title(f"min_cluster_size={mcs}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## Save chosen clustering back to atlas

Once you've picked the best `min_cluster_size`, set it below and run to update the parquet file.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# Set your chosen UMAP and clustering params here
BEST_NN = 15
BEST_MD = 0.1
BEST_MIN_CLUSTER_SIZE = 50

ATLAS_PATH = "./outputs/merged_1b_p10k/dashboard/features_atlas.parquet"

# Get the results
best_coords_2d = results[(BEST_NN, BEST_MD)]
best_cluster_ids = cluster_results[BEST_MIN_CLUSTER_SIZE]

# Update parquet
df = pq.read_table(ATLAS_PATH).to_pandas()
df["x"] = best_coords_2d[:, 0]
df["y"] = best_coords_2d[:, 1]

# Convert -1 (noise) to None for parquet compatibility
cluster_ids_safe = [int(c) if c >= 0 else None for c in best_cluster_ids]
df["cluster_id"] = cluster_ids_safe

pq.write_table(pa.Table.from_pandas(df, preserve_index=False), ATLAS_PATH)
print(f"Updated {ATLAS_PATH}")
print(f"  UMAP: n_neighbors={BEST_NN}, min_dist={BEST_MD}")
print(f"  HDBSCAN: min_cluster_size={BEST_MIN_CLUSTER_SIZE}")